# 🚲 Bike Demand Prediction

## Project Overview

This project analyzes historical bike-sharing data to understand patterns in bike demand and build a machine learning model to predict rental demand using time and weather information.

### Objectives
- Understand patterns in bike demand.
- Explore relationships between demand, time, and weather.
- Build a baseline regression model.
- Compare it with a non-linear machine learning model.
- Perform feature engineering and error analysis.
- Interpret the final model using feature importance.

## Workflow

**Data → Cleaning → EDA → Feature Engineering → Time-Based Split → Modeling → Evaluation → Error Analysis → Interpretation**


## 1. Imports

All libraries required for the project are loaded in one place so the notebook can be run from top to bottom without missing-import errors.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from IPython.display import display


## 6. Time-Based Train/Test Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 2. Data Loading

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00560/SeoulBikeData.csv"

df = pd.read_csv(
    url,
    encoding="unicode_escape"
)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.columns = [
    "date",
    "bike_count",
    "hour",
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall",
    "season",
    "holiday",
    "functioning_day"
]

## 4. Data Cleaning

In [ ]:
df.info()
df.describe()
df.isnull().sum()
df.nunique()

In [ ]:
df["date"] = pd.to_datetime(
    df["date"],
    dayfirst=True
)

In [ ]:
df["date"].dtype

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

## 5. Exploratory Data Analysis

In [ ]:
hourly_demand = (
    df.groupby("hour")["bike_count"]
    .mean()
)

hourly_demand.plot(
    kind="line",
    marker="o"
)

plt.title("Average Bike Demand by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Bikes Rented")
plt.show()

In [ ]:
season_demand = (
    df.groupby("season")["bike_count"]
    .mean()
    .sort_values(ascending=False)
)

season_demand.plot(
    kind="bar"
)

plt.title("Average Bike Demand by Season")
plt.xlabel("Season")
plt.ylabel("Average Bikes Rented")
plt.show()

In [ ]:
plt.scatter(
    df["temperature"],
    df["bike_count"]
)

plt.xlabel("Temperature")
plt.ylabel("Bike Count")
plt.title("Temperature vs Bike Demand")

plt.show()

In [ ]:
numeric_columns = [
    "bike_count",
    "hour",
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall"
]

correlation = df[numeric_columns].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
X = df.drop(
    columns=["bike_count", "date"]
)

y = df["bike_count"]

In [ ]:
df = df.sort_values("date")

In [ ]:
X = df.drop(
    columns=["bike_count", "date"]
)

y = df["bike_count"]

In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [ ]:
categorical_features = [
    "season",
    "holiday",
    "functioning_day"
]

numerical_features = [
    "hour",
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall",
    "year",
    "month",
    "day",
    "day_of_week"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

## 7. Baseline Model — Linear Regression

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(
    X_train,
    y_train
)

In [ ]:
linear_predictions = linear_model.predict(X_test)

## 9. Model Evaluation

In [ ]:
mae = mean_absolute_error(
    y_test,
    linear_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

r2 = r2_score(
    y_test,
    linear_predictions
)

print("Linear Regression")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

## 8. Random Forest Model

In [ ]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
random_forest.fit(
    X_train,
    y_train
)

In [ ]:
rf_predictions = random_forest.predict(
    X_test
)

In [ ]:
rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print("Random Forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

In [ ]:
gradient_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)
gradient_model.fit(
    X_train,
    y_train
)

In [ ]:
gradient_predictions = gradient_model.predict(
    X_test
)

In [ ]:
gradient_mae = mean_absolute_error(
    y_test,
    gradient_predictions
)

gradient_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gradient_predictions
    )
)

gradient_r2 = r2_score(
    y_test,
    gradient_predictions
)

print("Gradient Boosting")
print("MAE:", gradient_mae)
print("RMSE:", gradient_rmse)
print("R²:", gradient_r2)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    
    "MAE": [
        mae,
        rf_mae,
        gradient_mae
    ],
    
    "RMSE": [
        rmse,
        rf_rmse,
        gradient_rmse
    ],
    
    "R2": [
        r2,
        rf_r2,
        gradient_r2
    ]
})

results.sort_values(
    "RMSE"
)

## 13. Final Prediction Analysis

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    y_test.values[:200],
    label="Actual"
)

plt.plot(
    rf_predictions[:200],
    label="Predicted"
)

plt.title("Actual vs Predicted Bike Demand")

plt.xlabel("Test Observations")
plt.ylabel("Bike Count")

plt.legend()

plt.show()

In [ ]:
error_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": rf_predictions
})

error_df["error"] = (
    error_df["actual"]
    - error_df["predicted"]
)

error_df["absolute_error"] = (
    error_df["error"]
    .abs()
)

error_df.sort_values(
    "absolute_error",
    ascending=False
).head(10)

In [ ]:
new_data = pd.DataFrame({
    "hour": [18],
    "temperature": [25],
    "humidity": [50],
    "wind_speed": [2.0],
    "visibility": [1500],
    "dew_point": [15],
    "solar_radiation": [1.2],
    "rainfall": [0],
    "snowfall": [0],
    "season": ["Summer"],
    "holiday": ["No Holiday"],
    "functioning_day": ["Yes"],
    "year": [2018],
    "month": [7],
    "day": [15],
    "day_of_week": [6]
})

In [ ]:
prediction = random_forest.predict(
    new_data
)

print(
    "Predicted bikes:",
    round(prediction[0])
)

In [ ]:
import joblib

joblib.dump(
    random_forest,
    "bike_demand_model.pkl"
)

In [ ]:
loaded_model = joblib.load(
    "bike_demand_model.pkl"
)

In [ ]:
# How is bike demand distributed?
plt.figure(figsize=(8, 5))

sns.histplot(
    df["bike_count"],
    bins=40,
    kde=True
)

plt.title("Distribution of Bike Demand")
plt.xlabel("Number of Bikes Rented")
plt.ylabel("Frequency")

plt.show()

In [ ]:
hourly_demand = (
    df.groupby("hour")["bike_count"]
      .mean()
      .sort_values(ascending=False)
)

print(hourly_demand)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=hourly_demand.index,
    y=hourly_demand.values
)

plt.title("Average Bike Demand by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Bikes Rented")

plt.show()

In [ ]:
# Compare bike demand across days of the week

day_demand = (
    df.groupby("day_of_week")["bike_count"]
      .mean()
)

print(day_demand)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=day_demand.index,
    y=day_demand.values
)

plt.title("Average Bike Demand by Day of Week")
plt.xlabel("Day of Week (0 = Monday, 6 = Sunday)")
plt.ylabel("Average Bikes Rented")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="temperature",
    y="bike_count",
    alpha=0.3
)

plt.title("Temperature vs Bike Demand")
plt.xlabel("Temperature")
plt.ylabel("Bike Count")

plt.show()

In [ ]:
# How strongly is each numerical feature related to bike demand?

correlation_with_target = (
    df.corr(numeric_only=True)["bike_count"]
      .sort_values(ascending=False)
)

print(correlation_with_target)

In [ ]:
# Features = information the model can use
X = df.drop(columns=["bike_count", "date"])

# Target = what we want the model to predict
y = df["bike_count"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Make sure the data is ordered chronologically
df = df.sort_values("date")

# 80% of the data will be used for training
split_index = int(len(df) * 0.8)

# Earlier data → training
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]

# Later data → testing
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training period:")
print(df["date"].iloc[0], "to", df["date"].iloc[split_index - 1])

print("\nTesting period:")
print(df["date"].iloc[split_index], "to", df["date"].iloc[-1])

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical columns
numerical_features = [
    "hour",
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall",
    "year",
    "month",
    "day",
    "day_of_week"
]

# Categorical columns
categorical_features = [
    "season",
    "holiday",
    "functioning_day"
]

# Tell sklearn how each type of column should be processed
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Build a complete ML pipeline:
# 1. Preprocess the data
# 2. Train Linear Regression
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# Learn patterns ONLY from the training data
linear_model.fit(X_train, y_train)

# Predict bike demand for unseen test data
y_pred = linear_model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# MAE: average absolute prediction error
mae = mean_absolute_error(y_test, y_pred)

# RMSE: gives extra penalty to large errors
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R²: how much of the variation our model explains
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create the complete pipeline
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# Train only on the training data
random_forest.fit(X_train, y_train)

# Predict on unseen test data
rf_pred = random_forest.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_pred)
)

rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

In [ ]:
# Create a dataframe to inspect our model's predictions
error_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": rf_pred
})

# Calculate the prediction error
error_df["error"] = (
    error_df["actual"] - error_df["predicted"]
)

# Absolute error tells us how far off we were,
# regardless of whether we overpredicted or underpredicted
error_df["absolute_error"] = (
    error_df["error"].abs()
)

# Show the 10 biggest mistakes
error_df.sort_values(
    "absolute_error",
    ascending=False
).head(10)

In [ ]:
# Copy the test features
error_analysis = X_test.copy()

# Add the actual bike demand
error_analysis["actual"] = y_test.values

# Add our model's prediction
error_analysis["predicted"] = rf_pred

# Calculate how far the prediction was from reality
error_analysis["absolute_error"] = (
    error_analysis["actual"]
    - error_analysis["predicted"]
).abs()

# Show the biggest mistakes
error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(10)

## 10. Error Analysis

In [ ]:
hour_errors = (
    error_analysis
    .groupby("hour")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

print(hour_errors)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=hour_errors.index,
    y=hour_errors.values
)

plt.title("Average Prediction Error by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Absolute Error")

plt.show()

In [ ]:
# Average actual and predicted demand for each hour

hour_comparison = (
    error_analysis
    .groupby("hour")[["actual", "predicted"]]
    .mean()
)

print(hour_comparison)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    hour_comparison.index,
    hour_comparison["actual"],
    marker="o",
    label="Actual"
)

plt.plot(
    hour_comparison.index,
    hour_comparison["predicted"],
    marker="o",
    label="Predicted"
)

plt.title("Actual vs Predicted Bike Demand by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Bikes Rented")

plt.legend()

plt.show()

In [ ]:
season_errors = (
    error_analysis
    .groupby("season")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
)

print(season_errors)
plt.figure(figsize=(8, 5))

sns.barplot(
    x=season_errors.index,
    y=season_errors.values
)

plt.title("Average Prediction Error by Season")
plt.xlabel("Season")
plt.ylabel("Average Absolute Error")

plt.show()

## 11. Feature Engineering — Rush Hour

In [ ]:
# Create a rush-hour feature
df["is_rush_hour"] = df["hour"].isin(
    [7, 8, 9, 17, 18, 19]
).astype(int)

# Check the result
df[["hour", "is_rush_hour"]].drop_duplicates().sort_values("hour")

In [ ]:
# Add the new feature to training and testing data
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

X_train_fe["is_rush_hour"] = X_train_fe["hour"].isin(
    [7, 8, 9, 17, 18, 19]
).astype(int)

X_test_fe["is_rush_hour"] = X_test_fe["hour"].isin(
    [7, 8, 9, 17, 18, 19]
).astype(int)


# Automatically identify numerical and categorical columns
numeric_features_fe = X_train_fe.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features_fe = X_train_fe.select_dtypes(
    exclude=np.number
).columns.tolist()


# Build a new preprocessor
preprocessor_fe = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features_fe
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features_fe
        )
    ]
)


# Random Forest with the new feature
rf_feature_engineered = Pipeline(
    steps=[
        ("preprocessor", preprocessor_fe),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# Train
rf_feature_engineered.fit(
    X_train_fe,
    y_train
)


# Predict
rf_fe_pred = rf_feature_engineered.predict(
    X_test_fe
)

In [ ]:
# Evaluate the new model

fe_mae = mean_absolute_error(
    y_test,
    rf_fe_pred
)

fe_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_fe_pred
    )
)

fe_r2 = r2_score(
    y_test,
    rf_fe_pred
)

print("Random Forest + Rush Hour Feature")
print("----------------------------------")
print("MAE :", fe_mae)
print("RMSE:", fe_rmse)
print("R²  :", fe_r2)

## 12. Feature Importance

In [ ]:
# Get the trained Random Forest model
rf_model = rf_feature_engineered.named_steps["model"]

# Get feature names after preprocessing
feature_names = (
    rf_feature_engineered
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Create a table of feature importance
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
})

# Show the most important features
feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
)

feature_importance.head(15)

## 3. Data Understanding

In [ ]:
# Add prediction error to our test data

final_analysis = X_test_fe.copy()

final_analysis["actual"] = y_test.values
final_analysis["predicted"] = rf_fe_pred

final_analysis["error"] = (
    final_analysis["actual"]
    - final_analysis["predicted"]
)

final_analysis["absolute_error"] = (
    final_analysis["error"].abs()
)

final_analysis.head()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    final_analysis["actual"],
    final_analysis["predicted"],
    alpha=0.4
)

# Perfect prediction line
plt.plot(
    [final_analysis["actual"].min(),
     final_analysis["actual"].max()],
    [final_analysis["actual"].min(),
     final_analysis["actual"].max()],
    linestyle="--"
)

plt.title("Actual vs Predicted Bike Demand")
plt.xlabel("Actual Demand")
plt.ylabel("Predicted Demand")

plt.show()

## 14. High-Demand Error Analysis

In [ ]:
# How much does the model over/under-predict?

high_demand = final_analysis[
    final_analysis["actual"] >= 1500
]

print("High-demand observations:", len(high_demand))

print(
    "Average error:",
    high_demand["error"].mean()
)

print(
    "Average absolute error:",
    high_demand["absolute_error"].mean()
)

# 15. Final Findings & Conclusion

The analysis shows that bike demand varies strongly by time of day and is also associated with weather-related variables.

Linear Regression provided a baseline R² of approximately **0.45**. Random Forest substantially improved performance, reaching an R² of approximately **0.73**. Adding an `is_rush_hour` feature produced a small additional improvement, with the final model reaching an R² of approximately **0.736**, MAE of approximately **222 bikes**, and RMSE of approximately **315 bikes**.

Feature importance analysis identified **temperature** and **hour** as the strongest contributors to the Random Forest predictions, followed by solar radiation and humidity.

Error analysis showed that the model performs better for low and moderate demand levels but tends to underpredict extreme demand spikes. For the 233 test observations with actual demand of at least 1,500 bikes, the average error was approximately **+388 bikes**, indicating systematic underprediction in this high-demand group.

### Key Takeaways

1. Time of day is an important signal for bike demand.
2. Weather variables provide substantial predictive information.
3. Random Forest captures non-linear relationships better than Linear Regression.
4. Rush-hour feature engineering provides a modest improvement.
5. Extreme demand peaks remain the main weakness of the final model.

### Future Improvements

- Add richer time-based features.
- Explore lag and rolling-demand features where appropriate.
- Incorporate holiday and event information.
- Tune Random Forest hyperparameters and compare additional models.
- Investigate methods specifically suited to demand forecasting.
